In [3]:
import os
import math
import random
import re
import string
import math, random, re, gc, torch
import hashlib
from collections import deque
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F
from model import GPTModel
from tokenizer import CharTokenizer
from dataclass import Instance, Task
from registry import TASKS
from save_data import save_mixed_trace_file
from tqdm.auto import tqdm

In [ ]:

task_name = "word_index"

rho_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
model_seeds = [1000, 1500, 2200, 3000, 4000]

batch_size = 256
steps = 6000
learning_rate = 3e-4
min_learning_rate = 1e-5
warmup_steps = 600
weight_decay = 0.01
max_grad_norm = 1.0

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.05

val_seed = 99999
val_size = 1000
val_batch_size = 256

USE_COMPILE = False

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.set_float32_matmul_precision("high")

use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()

task = TASKS[task_name]
tokenizer = task.tokenizer

block_size = task.block_size
max_new_tokens = task.max_new_tokens
pad_id = tokenizer.pad_id
newline_id = tokenizer.newline_id
vocab_size = tokenizer.vocab_size

print("Device:", device)
print("BF16:", use_bf16)


with open(f"{task_name}/{task_name}_rho_0.txt", "r") as f:
    first_lines = [line.strip() for line in f if line.strip()]

train_prompts = {line.split(" ", 1)[0] for line in first_lines}
train_words = {prompt.split(";")[0] for prompt in train_prompts}

random.seed(val_seed)

val_instances = []
val_words = set()

while len(val_instances) < val_size:
    inst = task.sample()
    word = inst.prompt.split(";")[0]

    if word in train_words or word in val_words:
        continue

    val_words.add(word)
    val_instances.append(inst)

print("Validation examples:", len(val_instances))
print("Train/validation overlap:", len(train_words & val_words))

val_prompt_tensor = torch.tensor(
    [tokenizer.encode(inst.prompt) for inst in val_instances],
    dtype=torch.long,
    device=device
)


all_accuracies = {rho: [] for rho in rho_values}
all_separator_rates = {rho: [] for rho in rho_values}


for rho in rho_values:

    print("\n" + "=" * 70)
    print(f"ρ = {rho:.2f}")
    print("=" * 70)

    data_file = f"{task_name}/{task_name}_rho_{int(rho*100)}.txt"

    with open(data_file, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    encoded = []
    max_seq_len = 0

    for line in lines:
        prompt, continuation = line.split(" ", 1)

        p_ids = tokenizer.encode(prompt)
        t_ids = tokenizer.encode(" " + continuation + "\n")

        full = p_ids + t_ids

        x = full[:-1]
        y = full[1:]

        mask = [1.0 if i + 1 >= len(p_ids) else 0.0 for i in range(len(x))]

        encoded.append((x, y, mask))
        max_seq_len = max(max_seq_len, len(x))


    xs, ys, masks = [], [], []

    for x, y, mask in encoded:
        pad = max_seq_len - len(x)

        xs.append(x + [pad_id] * pad)
        ys.append(y + [pad_id] * pad)
        masks.append(mask + [0.0] * pad)


    xs = torch.tensor(xs, dtype=torch.long, device=device)
    ys = torch.tensor(ys, dtype=torch.long, device=device)
    masks = torch.tensor(masks, dtype=torch.float32, device=device)

    del encoded, lines


    for model_seed in model_seeds:

        print(f"\nρ={rho:.2f} | seed={model_seed}")

        random.seed(model_seed)
        torch.manual_seed(model_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(model_seed)


        base_model = GPTModel(
            vocab_size=vocab_size,
            block_size=block_size,
            pad_id=pad_id,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            dropout=dropout
        ).to(device)


        model = torch.compile(base_model) if USE_COMPILE and device == "cuda" else base_model


        try:
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=learning_rate,
                weight_decay=weight_decay,
                fused=(device == "cuda")
            )
        except:
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=learning_rate,
                weight_decay=weight_decay
            )


        model.train()

        pbar = tqdm(
            range(steps),
            desc=f"ρ={rho:.2f}, seed={model_seed}",
            leave=False
        )


        for step in pbar:

            if step < warmup_steps:
                lr = learning_rate * (step + 1) / warmup_steps
            else:
                decay = (step - warmup_steps) / (steps - warmup_steps)
                coeff = 0.5 * (1.0 + math.cos(math.pi * decay))
                lr = min_learning_rate + coeff * (learning_rate - min_learning_rate)

            for group in optimizer.param_groups:
                group["lr"] = lr


            ix = torch.randint(0, xs.shape[0], (batch_size,), device=device)

            xb = xs[ix]
            yb = ys[ix]
            mb = masks[ix]

            optimizer.zero_grad(set_to_none=True)


            if use_bf16:
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    _, loss = model(xb, targets=yb, mask=mb)
            else:
                _, loss = model(xb, targets=yb, mask=mb)


            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()


            if step % 100 == 0:
                pbar.set_postfix(loss=f"{loss.item():.4f}")


        base_model.eval()

        correct = 0
        separator_count = 0


        with torch.inference_mode():

            for start in range(0, val_size, val_batch_size):

                end = min(start + val_batch_size, val_size)

                context = val_prompt_tensor[start:end]

                out = base_model.generate(
                    context,
                    max_new_tokens=max_new_tokens,
                    stop_id=newline_id,
                    greedy=True
                )


                for row, inst in zip(out.tolist(), val_instances[start:end]):

                    text = tokenizer.decode(row)
                    generated = text[len(inst.prompt):].split("\n", 1)[0]

                    if ":" in generated:
                        separator_count += 1
                        answer = generated.rsplit(":", 1)[1]
                        nums = re.findall(r"\d+", answer)
                        pred = nums[0] if nums else None
                    else:
                        pred = None

                    correct += int(pred == inst.gold)


        val_acc = correct / val_size
        separator_rate = separator_count / val_size

        all_accuracies[rho].append(val_acc)
        all_separator_rates[rho].append(separator_rate)

        print(
            f"ρ={rho:.2f} | seed={model_seed} | "
            f"acc={val_acc*100:.2f}% | sep={separator_rate*100:.2f}%"
        )


        torch.save(
            base_model.state_dict(),
            f"{task_name}/{task_name}_rho_{int(rho*100)}_seed_{model_seed}.pt"
        )


        del model, base_model, optimizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    del xs, ys, masks

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Device: cuda
BF16: True
Validation examples: 1000
Train/validation overlap: 0

ρ = 0.00

ρ=0.00 | seed=1000


ρ=0.00, seed=1000:   0%|          | 0/6000 [00:00<?, ?it/s]

ρ=0.00 | seed=1000 | acc=11.20% | sep=100.00%

ρ=0.00 | seed=1500


ρ=0.00, seed=1500:   0%|          | 0/6000 [00:00<?, ?it/s]

ρ=0.00 | seed=1500 | acc=11.30% | sep=100.00%

ρ=0.00 | seed=2200


...

ρ=0.20, seed=3000:   0%|          | 0/6000 [00:00<?, ?it/s]

ρ=0.20 | seed=3000 | acc=9.80% | sep=100.00%

ρ=0.20 | seed=4000


ρ=0.20, seed=4000:   0%|          | 0/6000 [00:00<?, ?it/s]

ρ=0.20 | seed=4000 | acc=9.60% | sep=100.00%

ρ = 0.30

ρ=0.30 | seed=1000


ρ=0.30, seed=1000:   0%|          | 0/6000 [00:00<?, ?it/s]

In [ ]:
mean_acc = []
std_acc = []

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

for rho in rho_values:

    values = np.array(all_accuracies[rho]) * 100

    mean = values.mean()
    std = values.std(ddof=1)

    mean_acc.append(mean)
    std_acc.append(std)

    print(
        f"ρ={rho:.2f} | "
        f"mean={mean:.2f}% | "
        f"std={std:.2f}% | "
        f"runs={values}"
    )

In [ ]:
mean_acc = []
std_acc = []

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

for rho in rho_values:

    values = np.array(all_accuracies[rho]) * 100

    mean = values.mean()
    std = values.std(ddof=1)

    mean_acc.append(mean)
    std_acc.append(std)

    print(
        f"ρ={rho:.2f} | "
        f"mean={mean:.2f}% | "
        f"std={std:.2f}% | "
        f"runs={values}"
    )